# Multimodal Chest X-Ray Classification
## Image (EfficientNet) + Text (ClinicalBERT) + Label Cleaning
### DATA 2026

In [ ]:
# Install required packages
import subprocess
subprocess.run(['pip', 'install', 'transformers', 'sentencepiece', '-q'])
print('Packages ready.')

In [ ]:
import os, warnings, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, roc_curve, auc as sk_auc
from sklearn.model_selection import train_test_split
from PIL import Image
from transformers import AutoTokenizer, AutoModel
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# Paths
IMAGE_DIR    = Path('/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset/official_data_iccv_final/files')
REPORTS_DIR  = Path('/kaggle/input/datasets/rifathossain47/mimic-cxr-official-csv/mimic-cxr-reports/files')
CHEXPERT_CSV = Path('/kaggle/input/datasets/rifathossain47/mimic-cxr-official-csv/mimic-cxr-2.0.0-chexpert.csv')
SPLIT_CSV    = Path('/kaggle/input/datasets/rifathossain47/mimic-cxr-official-csv/mimic-cxr-2.0.0-split.csv')

print(f'IMAGE_DIR:    {IMAGE_DIR.exists()}')
print(f'REPORTS_DIR:  {REPORTS_DIR.exists()}')
print(f'CHEXPERT_CSV: {CHEXPERT_CSV.exists()}')
print(f'SPLIT_CSV:    {SPLIT_CSV.exists()}')

# Config
DISEASES    = ['Pleural Effusion', 'Atelectasis', 'Cardiomegaly', 'Pneumonia', 'Pneumothorax']
IMG_SIZE    = 224
BATCH_SIZE  = 16   # smaller because multimodal uses more memory
LR          = 1e-4
EPOCHS      = 5
PATIENCE    = 3
MAX_LEN     = 256  # max text tokens
CLEAN_HIGH  = 0.70
CLEAN_LOW   = 0.20
N_TRAIN     = 10000
N_VAL       = 2000
N_TEST      = 2000

print('Config ready. Diseases:', DISEASES)

In [ ]:
# Load CSV data
chexpert = pd.read_csv(CHEXPERT_CSV)
split_df = pd.read_csv(SPLIT_CSV)
df = chexpert.merge(split_df, on=['subject_id','study_id'], how='left')

for col in DISEASES:
    if col in df.columns:
        df[col] = df[col].replace(-1,0).fillna(0).astype(int)

DISEASES = [d for d in DISEASES if d in df.columns]
df = df[['subject_id','study_id','dicom_id','split'] + DISEASES].copy()
print(f'Total rows: {len(df):,}')

In [ ]:
# Build image and report paths
def build_img_path(row):
    subj  = str(int(row['subject_id']))
    study = str(int(row['study_id']))
    dicom = str(row['dicom_id'])
    return IMAGE_DIR / ('p'+subj[:2]) / ('p'+subj) / ('s'+study) / (dicom+'.jpg')

def build_report_path(row):
    subj  = str(int(row['subject_id']))
    study = str(int(row['study_id']))
    return REPORTS_DIR / ('p'+subj[:2]) / ('p'+subj) / ('s'+study+'.txt')

df['img_path']    = df.apply(build_img_path, axis=1)
df['report_path'] = df.apply(build_report_path, axis=1)

# Check samples
print('Image path check:')
for p in df['img_path'].head(3):
    print(f'  {p.exists()} — {p.name}')
print('Report path check:')
for p in df['report_path'].head(3):
    print(f'  {p.exists()} — {p.name}')

In [ ]:
# Read report text
def read_report(path):
    try:
        text = path.read_text(encoding='utf-8', errors='ignore')
        # Extract findings and impression sections
        text = re.sub(r'\s+', ' ', text).strip()
        # Keep only relevant sections
        for section in ['FINDINGS:', 'IMPRESSION:', 'CONCLUSION:']:
            if section in text.upper():
                idx = text.upper().find(section)
                text = text[idx:]
                break
        return text[:512] if text else 'no findings'
    except:
        return 'no findings'

# Test reading
sample_report = df[df['report_path'].apply(lambda p: p.exists())].iloc[0]
print('Sample report text:')
print(read_report(sample_report['report_path'])[:300])

In [ ]:
# Patient-level split
patients = df['subject_id'].unique()
tr, tmp = train_test_split(patients, test_size=0.30, random_state=SEED)
vl, ts  = train_test_split(tmp, test_size=0.50, random_state=SEED)
train_df = df[df['subject_id'].isin(tr)].copy()
val_df   = df[df['subject_id'].isin(vl)].copy()
test_df  = df[df['subject_id'].isin(ts)].copy()

# Subsample
train_df = train_df.sample(min(N_TRAIN, len(train_df)), random_state=SEED).reset_index(drop=True)
val_df   = val_df.sample(min(N_VAL,   len(val_df)),   random_state=SEED).reset_index(drop=True)
test_df  = test_df.sample(min(N_TEST,  len(test_df)),  random_state=SEED).reset_index(drop=True)

# Filter to existing images
print('Filtering existing images...')
train_df = train_df[train_df['img_path'].apply(lambda p: p.exists())].reset_index(drop=True)
val_df   = val_df[val_df['img_path'].apply(lambda p: p.exists())].reset_index(drop=True)
test_df  = test_df[test_df['img_path'].apply(lambda p: p.exists())].reset_index(drop=True)

# Fix empty test
if len(test_df) == 0:
    split_point = len(val_df) // 2
    test_df = val_df.iloc[split_point:].reset_index(drop=True)
    val_df  = val_df.iloc[:split_point].reset_index(drop=True)

print(f'Train:{len(train_df):,} Val:{len(val_df):,} Test:{len(test_df):,}')

In [ ]:
# Load ClinicalBERT tokenizer
print('Loading ClinicalBERT...')
BERT_MODEL = 'emilyalsentzer/Bio_ClinicalBERT'
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
print('Tokenizer ready.')

In [ ]:
# Multimodal Dataset
class MultimodalCXRDataset(Dataset):
    def __init__(self, df, diseases, tokenizer, max_len=256, transform=None):
        self.df = df.reset_index(drop=True)
        self.diseases  = diseases
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Image
        try: img = Image.open(row['img_path']).convert('RGB')
        except: img = Image.new('RGB',(IMG_SIZE,IMG_SIZE),(128,128,128))
        if self.transform: img = self.transform(img)

        # Text
        text = read_report(row['report_path'])
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        input_ids      = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)

        # Labels
        labels = torch.FloatTensor(row[self.diseases].values.astype(float))

        return img, input_ids, attention_mask, labels, idx


# Transforms
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE+32, IMG_SIZE+32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_loader = DataLoader(MultimodalCXRDataset(train_df,DISEASES,tokenizer,MAX_LEN,train_tf),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(MultimodalCXRDataset(val_df,  DISEASES,tokenizer,MAX_LEN,val_tf),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(MultimodalCXRDataset(test_df, DISEASES,tokenizer,MAX_LEN,val_tf),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Loaders ready — Train:{len(train_loader)} Val:{len(val_loader)} Test:{len(test_loader)} batches')

In [ ]:
# Multimodal Model: EfficientNet + ClinicalBERT
class MultimodalCXRModel(nn.Module):
    """
    Combines:
    - EfficientNet-B3: visual features from X-ray image
    - ClinicalBERT: semantic features from radiology report
    - Fusion layer: combines both modalities
    """
    def __init__(self, num_classes, bert_model_name='emilyalsentzer/Bio_ClinicalBERT'):
        super().__init__()

        # Image encoder — EfficientNet-B3
        try:
            from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
            eff = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)
        except:
            eff = models.efficientnet_b3(pretrained=True)
        self.image_encoder = nn.Sequential(*list(eff.children())[:-1])
        self.image_dim = 1536  # EfficientNet-B3 output dim

        # Text encoder — ClinicalBERT
        self.text_encoder = AutoModel.from_pretrained(bert_model_name)
        self.text_dim = 768  # BERT hidden size

        # Fusion layers
        self.image_proj = nn.Sequential(
            nn.Linear(self.image_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.text_proj = nn.Sequential(
            nn.Linear(self.text_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, images, input_ids, attention_mask):
        # Image features
        img_feat = self.image_encoder(images)
        img_feat = img_feat.flatten(1)
        img_feat = self.image_proj(img_feat)

        # Text features — use [CLS] token
        txt_out  = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        txt_feat = txt_out.last_hidden_state[:, 0, :]  # CLS token
        txt_feat = self.text_proj(txt_feat)

        # Fusion — concatenate
        fused = torch.cat([img_feat, txt_feat], dim=1)  # (B, 1024)

        return self.classifier(fused)


# Test model
test_model = MultimodalCXRModel(len(DISEASES)).to(DEVICE)
total = sum(p.numel() for p in test_model.parameters())
print(f'Multimodal model params: {total:,}')
del test_model; torch.cuda.empty_cache()

In [ ]:
# Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__(); self.gamma=gamma; self.alpha=alpha
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        bce   = nn.functional.binary_cross_entropy_with_logits(logits,targets,reduction='none')
        pt    = targets*probs+(1-targets)*(1-probs)
        loss  = ((1-pt)**self.gamma)*bce
        if self.alpha is not None: loss = self.alpha.to(logits.device)*loss
        return loss.mean()

pos = train_df[DISEASES].sum().values
neg = len(train_df)-pos
w   = torch.FloatTensor(neg/(pos+1e-6)); w=w/w.mean()
criterion = FocalLoss(alpha=w, gamma=2.0)
print('Loss ready. Class weights:'); [print(f'  {d}: {v:.3f}') for d,v in zip(DISEASES,w)]

In [ ]:
# Training functions
def compute_metrics(y_true, y_pred, thr=0.5):
    y_bin = (y_pred>thr).astype(int)
    res, auc_list = {}, []
    for i,d in enumerate(DISEASES):
        a = roc_auc_score(y_true[:,i],y_pred[:,i]) if y_true[:,i].sum()>0 else float('nan')
        auc_list.append(a); res[f'AUC_{d}']=a
    res['macro_auc']  = np.nanmean(auc_list)
    res['macro_f1']   = f1_score(y_true,y_bin,average='macro',zero_division=0)
    res['macro_prec'] = precision_score(y_true,y_bin,average='macro',zero_division=0)
    res['macro_rec']  = recall_score(y_true,y_bin,average='macro',zero_division=0)
    return res

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, all_preds, all_labels = 0,[],[]
    for i,(imgs,input_ids,attn_mask,labels,_) in enumerate(loader):
        imgs       = imgs.to(device)
        input_ids  = input_ids.to(device)
        attn_mask  = attn_mask.to(device)
        labels     = labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs, input_ids, attn_mask)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        with torch.no_grad(): all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.cpu().numpy())
        if i%10==0: print(f'  Batch {i}/{len(loader)} loss={loss.item():.4f}',end='\r')
    ap=np.vstack(all_preds); al=np.vstack(all_labels)
    try: auc=roc_auc_score(al,ap,average='macro')
    except: auc=0.0
    return total_loss/len(loader), auc

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss,all_preds,all_labels = 0,[],[]
    for imgs,input_ids,attn_mask,labels,_ in loader:
        imgs      = imgs.to(device)
        input_ids = input_ids.to(device)
        attn_mask = attn_mask.to(device)
        labels    = labels.to(device)
        logits = model(imgs, input_ids, attn_mask)
        total_loss += criterion(logits,labels).item()
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    ap=np.vstack(all_preds); al=np.vstack(all_labels)
    return total_loss/len(loader), compute_metrics(al,ap), ap, al

def train_model(model, tr_loader, vl_loader, optimizer, scheduler, criterion, epochs, patience, device, name):
    history=defaultdict(list); best_auc,pc,best_state=0,0,None
    for epoch in range(1,epochs+1):
        print(f'\nEpoch {epoch}/{epochs} [{name}]')
        tl,ta=train_epoch(model,tr_loader,optimizer,criterion,device)
        vl,vm,_,_=evaluate(model,vl_loader,criterion,device)
        scheduler.step(vm['macro_auc'])
        history['train_loss'].append(tl); history['val_loss'].append(vl)
        history['train_auc'].append(ta);  history['val_auc'].append(vm['macro_auc'])
        print(f'  Train Loss={tl:.4f} AUC={ta:.4f}')
        print(f'  Val   Loss={vl:.4f} AUC={vm["macro_auc"]:.4f} F1={vm["macro_f1"]:.4f}')
        if vm['macro_auc']>best_auc:
            best_auc=vm['macro_auc']
            best_state={k:v.clone() for k,v in model.state_dict().items()}
            torch.save(best_state,f'/kaggle/working/{name}_best.pt')
            print(f'  ✓ Best (AUC={best_auc:.4f})')
            pc=0
        else:
            pc+=1
            if pc>=patience: print(f'  Early stopping'); break
    if best_state: model.load_state_dict(best_state)
    return history

print('Functions ready.')

In [ ]:
# Train initial model (for label cleaning)
print('=== Training Initial Multimodal Model ===')
initial_model = MultimodalCXRModel(len(DISEASES)).to(DEVICE)

# Different LR for pretrained vs new layers
optimizer = optim.AdamW([
    {'params': initial_model.image_encoder.parameters(), 'lr': 1e-5},
    {'params': initial_model.text_encoder.parameters(),  'lr': 1e-5},
    {'params': initial_model.image_proj.parameters(),    'lr': LR},
    {'params': initial_model.text_proj.parameters(),     'lr': LR},
    {'params': initial_model.classifier.parameters(),    'lr': LR},
], weight_decay=1e-5)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
initial_history = train_model(initial_model, train_loader, val_loader, optimizer, scheduler,
                               criterion, EPOCHS, PATIENCE, DEVICE, 'initial_multimodal')
print('Initial model done.')

In [ ]:
# Label Cleaning
@torch.no_grad()
def get_predictions(model, df, diseases, device):
    ds = MultimodalCXRDataset(df, diseases, tokenizer, MAX_LEN, val_tf)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    model.eval(); all_preds=[]
    for imgs,input_ids,attn_mask,_,_ in dl:
        logits = model(imgs.to(device), input_ids.to(device), attn_mask.to(device))
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
    return np.vstack(all_preds)

def clean_labels(train_df, preds, diseases, high=0.70, low=0.20):
    cleaned=train_df.copy(); stats={}
    for i,d in enumerate(diseases):
        orig=cleaned[d].values.copy(); p=preds[:,i]
        fn=(orig==0)&(p>high); fp=(orig==1)&(p<low)
        cleaned.loc[fn,d]=1; cleaned.loc[fp,d]=0
        stats[d]={'fn_flipped':int(fn.sum()),'fp_flipped':int(fp.sum()),
                  'total':int(fn.sum()+fp.sum()),
                  'rate_pct':round((fn.sum()+fp.sum())/len(orig)*100,2)}
    return cleaned, stats

print('Getting predictions for label cleaning...')
train_preds = get_predictions(initial_model, train_df, DISEASES, DEVICE)
cleaned_train_df, noise_stats = clean_labels(train_df, train_preds, DISEASES, CLEAN_HIGH, CLEAN_LOW)

print('\n=== LABEL NOISE ANALYSIS ===')
print(f'{"Disease":30s} {"0→1":>8} {"1→0":>8} {"Total":>8} {"Rate%":>8}')
print('-'*60)
total_flipped=0
for d,s in noise_stats.items():
    print(f'{d:30s} {s["fn_flipped"]:>8} {s["fp_flipped"]:>8} {s["total"]:>8} {s["rate_pct"]:>7.2f}%')
    total_flipped+=s['total']
print(f'Total corrected: {total_flipped:,}')

In [ ]:
# Train final model on cleaned labels
cleaned_loader = DataLoader(
    MultimodalCXRDataset(cleaned_train_df, DISEASES, tokenizer, MAX_LEN, train_tf),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)

print('=== Training Final Multimodal Model (Cleaned Labels) ===')
final_model = MultimodalCXRModel(len(DISEASES)).to(DEVICE)
final_optimizer = optim.AdamW([
    {'params': final_model.image_encoder.parameters(), 'lr': 1e-5},
    {'params': final_model.text_encoder.parameters(),  'lr': 1e-5},
    {'params': final_model.image_proj.parameters(),    'lr': LR},
    {'params': final_model.text_proj.parameters(),     'lr': LR},
    {'params': final_model.classifier.parameters(),    'lr': LR},
], weight_decay=1e-5)
final_scheduler = optim.lr_scheduler.ReduceLROnPlateau(final_optimizer, mode='max', factor=0.5, patience=2)
final_history = train_model(final_model, cleaned_loader, val_loader, final_optimizer, final_scheduler,
                             criterion, EPOCHS, PATIENCE, DEVICE, 'final_multimodal')

In [ ]:
# Final Results
print('=== BASELINE vs MULTIMODAL + LABEL CLEANING ===')
_, baseline_results, _, _  = evaluate(initial_model, test_loader, criterion, DEVICE)
_, final_results,    preds, labels = evaluate(final_model, test_loader, criterion, DEVICE)

print(f'\n{"Metric":35s} {"Image Only":>12} {"Multimodal+Clean":>18} {"Δ":>8}')
print('-'*76)
for name,key in [("Macro AUC","macro_auc"),("Macro F1","macro_f1"),("Precision","macro_prec"),("Recall","macro_rec")]:
    b=baseline_results[key]; f=final_results[key]
    print(f'{name:35s} {b:>12.4f} {f:>18.4f} {f-b:>+8.4f}')

print(f'\n{"Disease":30s} {"Baseline":>12} {"Multimodal":>12} {"Δ":>8}')
print('-'*65)
for d in DISEASES:
    b=baseline_results.get(f'AUC_{d}',0)
    f=final_results.get(f'AUC_{d}',0)
    print(f'{d:30s} {b:>12.4f} {f:>12.4f} {f-b:>+8.4f}')

In [ ]:
# Results chart
fig,axes=plt.subplots(1,2,figsize=(14,5))
x=np.arange(len(DISEASES)); w=0.35
bl_aucs=[baseline_results.get(f'AUC_{d}',0) for d in DISEASES]
fn_aucs=[final_results.get(f'AUC_{d}',0)    for d in DISEASES]
axes[0].bar(x-w/2, bl_aucs, w, label='Image Only (baseline)', color='#2196F3', alpha=0.85)
axes[0].bar(x+w/2, fn_aucs, w, label='Multimodal + Cleaning', color='#4CAF50', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(DISEASES,rotation=30,ha='right')
axes[0].set_ylabel('AUC-ROC'); axes[0].set_ylim(0.5,1.0)
axes[0].set_title('Per-Class AUC Comparison',fontsize=13,fontweight='bold')
axes[0].legend()

mnames=['Macro AUC','Macro F1','Precision','Recall']
mkeys=['macro_auc','macro_f1','macro_prec','macro_rec']
xm=np.arange(len(mnames))
axes[1].bar(xm-w/2,[baseline_results[k] for k in mkeys],w,label='Image Only',color='#2196F3',alpha=0.85)
axes[1].bar(xm+w/2,[final_results[k]    for k in mkeys],w,label='Multimodal+Clean',color='#4CAF50',alpha=0.85)
axes[1].set_xticks(xm); axes[1].set_xticklabels(mnames)
axes[1].set_ylim(0,1.0); axes[1].set_title('Overall Metrics',fontsize=13,fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.savefig('/kaggle/working/multimodal_comparison.png',dpi=150,bbox_inches='tight')
plt.show(); print('Saved: multimodal_comparison.png')

In [ ]:
# ROC Curves
fig,axes=plt.subplots(1,len(DISEASES),figsize=(5*len(DISEASES),5))
for ax,disease in zip(axes,DISEASES):
    di=DISEASES.index(disease)
    for name,p,color in [('Baseline',np.vstack([baseline_results.get(f'AUC_{d}',0) for d in DISEASES]),'#2196F3'),
                          ('Multimodal',preds,'#4CAF50')]:
        if name=='Multimodal':
            fpr,tpr,_=roc_curve(labels[:,di],preds[:,di])
            a=sk_auc(fpr,tpr)
            ax.plot(fpr,tpr,color=color,lw=2,label=f'{name} ({a:.3f})')
    ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
    ax.set_title(disease,fontsize=11,fontweight='bold')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.legend(fontsize=8); ax.grid(True,alpha=0.3)
plt.suptitle('ROC Curves — Multimodal Model',fontsize=13,fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/roc_curves_multimodal.png',dpi=150,bbox_inches='tight')
plt.show(); print('Saved: roc_curves_multimodal.png')

## Output Files:
- `multimodal_comparison.png` → Figure 1
- `roc_curves_multimodal.png` → Figure 2
- `initial_multimodal_best.pt` → baseline model
- `final_multimodal_best.pt` → final model